# ***Predictive Modeling***

To identify beneficiaries at elevated risk of healthcare cost burden, a binary classification model was developed for two outcomes: **high out-of-pocket (OOP) burden** and **high payer cost burden**. 3 modeling approaches—**logistic regression, random forest, and XGBoost**—are evaluated to compare interpretable and nonlinear machine learning methods.

The prediction framework accounts for the information that would realistically be available at each point in time:

* **2008:** Predictions use baseline beneficiary demographics and chronic disease information.
* **2009:** Predictions incorporate 2008 historical information alongside beneficiary characteristics.
* **2010:** Predictions incorporate cumulative information from **2008 and 2009**, allowing the models to leverage a longer history of beneficiary utilization, disease burden, and spending.

This temporal structure is designed to prevent information leakage by ensuring that predictors come from information available **before the outcome being predicted**.

The analysis therefore addresses two related questions:

1. **Can we identify beneficiaries who will experience high OOP or payer cost burden?**
2. **Does incorporating longitudinal beneficiary history improve our ability to identify future high-cost beneficiaries?**

Model performance is evaluated using discrimination and classification metrics, while feature-level interpretation is used to examine which demographic, clinical, utilization, and historical cost factors contribute most to high-cost predictions.


In [1]:
import numpy as np
import pandas as pd
import pickle
import os
import shutil
from src.modeling import *
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
patient = pd.read_csv("cleaned_data/patient_final.csv")
patient.head()

,DESYNPUF_ID,BENE_ESRD_IND,BENE_COUNTY_CD,PART_A_INSUR_CVRAGE_TOT_MONS,PART_B_INSUR_CVRAGE_TOT_MONS,HMO_INSUR_CVRAGE_TOT_MONS,PART_D_INSUR_CVRAGE_TOT_MONS,year,SEX,RACE,STATE,age,age_band,IP_bill,OP_bill,Carrier_bill,num_carrier_claims,avg_carrier_claim_duration,max_carrier_claim_duration,total_carrier_claim_duration,avg_carrier_rejected_amt,max_carrier_rejected_amt,total_carrier_rejected_amt,avg_carrier_claim_rejected_rate (%),max_carrier_claim_rejected_rate (%),avg_carrier_reimb_allowed_diff,min_carrier_reimb_allowed_diff,max_carrier_reimb_allowed_diff,total_carrier_reimb_allowed_diff,max_carrier_chronic_code_counts,avg_carrier_insurance_cost,max_carrier_insurance_cost,total_carrier_insurance_cost,avg_carrier_beneficiary_cost,max_carrier_beneficiary_cost,total_carrier_beneficiary_cost,has_Additional endocrine/metabolic burden,has_COPD,has_Administrative_screening_aftercare,has_Cancer,has_Injury/Frailty Burden,has_Additional cardiovascular burden,has_Additional mental health burden,has_Additional Musculoskeletal Burden,has_Infectious disease burden,has_Stroke/TIA,has_Additional neurologic burden,has_Diabetes,has_Additional renal burden,has_Genitourinary (non-renal) burden,has_Sensory Burden,has_Additional Hematologic burden,has_Rheumatoid Arthritis/Osteoarthritis,has_Depression,has_Heart Failure,has_CKD,has_Additional GI/Hepatic burden,num_ip_claims,avg_ip_claim_duration,max_ip_claim_duration,total_ip_claim_duration,avg_ip_pmt,max_ip_pmt,total_ip_pmt,max_ip_chronic_code_counts,avg_inpatient_insurance_cost,max_inpatient_insurance_cost,total_inpatient_insurance_cost,avg_inpatient_beneficiary_cost,max_inpatient_beneficiary_cost,total_inpatient_beneficiary_cost,num_op_claims,avg_op_claim_duration,max_op_claim_duration,total_op_claim_duration,avg_op_pmt,max_op_pmt,total_op_pmt,max_op_chronic_code_counts,avg_outpatient_insurance_cost,max_outpatient_insurance_cost,total_outpatient_insurance_cost,avg_outpatient_beneficiary_cost,max_outpatient_beneficiary_cost,total_outpatient_beneficiary_cost,num_pde_claims,total_days_medsupplied,avg_days_supplied_per_pdeclaim,max_days_supplied_per_pdeclaim,total_medoop,avg_medoop_per_claim,max_medoop_per_claim,total_raw_drug_cost,avg_raw_drug_cost_per_claim,max_raw_drug_cost_per_claim,has_carrier_record,has_ip_record,has_op_record,has_pde_record,num_service_types,ip_bill_diff,op_bill_diff,total_inpatient_insurance_cost_adj,total_outpatient_insurance_cost_adj,claim_processing_burden,payer_care_cost,out_of_pocket_care_cost,total_financial_impact,High_Payer_Cost,High_OOP_Burden
0,000102649ED5601B,0,290,12,12,0,12,2008,Female,White,VA,77,75-84,0.0,450.0,1030.0,16.0,0 days 01:30:00,1 days,1 days,0.000000,0.0,0.0,0.000000,0.0,22.500000,0.0,210.0,360.0,3.0,63.750000,290.0,1020.0,12.500000,50.0,200.0,1.0,1,1.0,1,1.0,1.0,1.0,1.0,1.0,1,1.0,1,1.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0 days 00:00:00,0 days,0 days,150.000000,320.0,450.0,1.0,120.000000,300.0,360.0,30.000000,70.0,90.0,21.0,760.0,36.190476,90.0,70.0,3.333333,10.0,500.0,23.809524,120.0,True,False,True,True,3.0,0.0,0.0,0.0,360.0,0.0,1880.0,360.0,2240.0,0,0
1,0002278C944E240A,1,250,12,12,0,12,2008,Female,White,NC,72,65-74,0.0,4120.0,6230.0,42.0,0 days 17:08:34.285714285,9 days,30 days,2.142857,50.0,90.0,4.878049,100.0,25.000000,0.0,470.0,1050.0,4.0,130.714280,750.0,5490.0,32.142857,310.0,1350.0,1.0,1,1.0,1,1.0,1.0,1.0,1.0,1.0,0,1.0,1,1.0,1.0,1.0,1.0,1,1,1,1,1.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,1 days 03:41:32.307692307,13 days,15 days,316.923077,1050.0,4120.0,7.0,199.230769,800.0,2590.0,117.692308,1000.0,1530.0,47.0,1210.0,25.744681,90.0,440.0,9.361702,170.0,3580.0,76.170213,570.0,True,False,True,True,3.0,0.0,0.0,0.0,2590.0,90.0,11660.0,3320.0,14980.0,0,1
2,000330E625C93700,0,420,12,12,0,0,2008,Female,White,AR,75,75-84,0.0,0.0,420.0,4.0,0 days 00:00:00,0 days,0 days,0.000000,0.0,0.0,0.000000,0.0,30.000000,0.0,60.0,120.0,1.0,67.500000,90.0,270.0,37.500000,70.0,150.0,0.0,0,0.0,0,0.0,1.0,0.0,

In [3]:
patient.groupby("year")[
    [
        "payer_care_cost",
        "out_of_pocket_care_cost",
        "total_medoop",
        "total_raw_drug_cost",
        "total_carrier_insurance_cost"
    ]
].agg(["mean", "median", "max"])

payer_care_cost                   out_of_pocket_care_cost         \
                mean  median       max                    mean median   
year                                                                    
2008     5314.272665  1990.0  197740.0              985.040776  490.0   
2009     5687.295894  2750.0  171020.0             1088.280348  670.0   
2010     3331.841783  1670.0  151680.0              653.572111  400.0   

              total_medoop                total_raw_drug_cost                  \
          max         mean median     max                mean median      max   
year                                                                            
2008  42974.0    177.67976   50.0  2490.0         1060.589280  270.0  15690.0   
2009  40902.0    197.86768   80.0  2290.0         1175.498656  440.0  13540.0   
2010  35970.0    126.67884   40.0  1750.0          755.486303  250.0  11150.0   

     total_carrier_insurance_cost                  
                             mean median      max  
year                                               
2008                  1269.084474  670.0  21420.0  
2009                  1456.333748  980.0  19050.0  
2010                   916.378835  590.0  10720.0

In [4]:
patient.groupby("year")[
    ["High_Payer_Cost", "High_OOP_Burden"]
].mean()

,High_Payer_Cost,High_OOP_Burden
year,,
2008,0.099921,0.099776
2009,0.099948,0.099722
2010,0.099939,0.099975


# **Part 1: Logistic Regression**

In [5]:
all_lr_results = []

for year in [2008, 2009, 2010]:
    X, y = build_features(patient, year)
    X = fix_feature_dtypes(X)
    results = run_logistic_regression_experiment(X, y)
    results["year"] = year
    all_lr_results.append(results)

lr_results_df = pd.concat(all_lr_results,ignore_index=True)
lr_results_df

Accuracy: 0.8256
ROC-AUC: 0.6646

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.88      0.90     19964
           1       0.25      0.37      0.30      2216

    accuracy                           0.83     22180
   macro avg       0.59      0.62      0.60     22180
weighted avg       0.86      0.83      0.84     22180


Confusion Matrix
[[17486  2478]
 [ 1390   826]]
Accuracy: 0.8348
ROC-AUC: 0.6709

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.89      0.91     19967
           1       0.27      0.38      0.32      2213

    accuracy                           0.83     22180
   macro avg       0.60      0.63      0.61     22180
weighted avg       0.86      0.83      0.85     22180


Confusion Matrix
[[17672  2295]
 [ 1370   843]]
Accuracy: 0.7391
ROC-AUC: 0.8333

Classification Report
              precision    recall  f1-score   support

           0       0.96  

,target,accuracy,roc_auc,year
0,High_Payer_Cost,0.825609,0.664638,2008
1,High_OOP_Burden,0.834761,0.670921,2008
2,High_Payer_Cost,0.739089,0.833312,2009
3,High_OOP_Burden,0.804599,0.898800,2009
4,High_Payer_Cost,0.673399,0.774274,2010
5,High_OOP_Burden,0.704509,0.814079,2010


Historical utilization and cost information substantially improved prediction relative to a demographic baseline, with the strongest predictive performance observed when one year of recent history was available. Adding an additional year of historical information did not yield proportional gains, suggesting that recent beneficiary behavior may provide more predictive value than simply accumulating additional historical measures.

The 2010 claims data also exhibited a substantial downward shift in payer and beneficiary cost distributions despite similar high-cost classification rates, indicating that year-to-year differences in the underlying cost distribution may influence model performance.

# **Part 2: Random Forest**

# **Part 3: XGBoost**

# **Part 4: Model Comparison**